In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/equal-neutral-submit-version/train_equal.csv
/kaggle/input/equal-neutral-submit-version/test_equal.csv
/kaggle/input/xlnet-best-model/config.json
/kaggle/input/xlnet-best-model/training_args.bin
/kaggle/input/xlnet-best-model/model.safetensors
/kaggle/input/best-electra-model/config.json
/kaggle/input/best-electra-model/training_args.bin
/kaggle/input/best-electra-model/model.safetensors
/kaggle/input/best-roberta-model/config.json
/kaggle/input/best-roberta-model/training_args.bin
/kaggle/input/best-roberta-model/model.safetensors


In [2]:
import torch
import numpy as np
import pandas as pd
import re
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
from tqdm import tqdm

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
model_configs = [
    ("/kaggle/input/best-electra-model", "google/electra-base-discriminator", 128),
    ("/kaggle/input/best-roberta-model", "roberta-base", 128),
    ("/kaggle/input/xlnet-best-model", "xlnet-base-cased", 128)
]

In [4]:
models = [AutoModelForSequenceClassification.from_pretrained(path).to(device) for path, _, _ in model_configs]
tokenizers = [AutoTokenizer.from_pretrained(base_name) for _, base_name, _ in model_configs]
max_lengths = [max_len for _, _, max_len in model_configs]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

In [5]:
def preprocess_text(text) :
    return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE).lower()

def prepare_undersampled_data(df) :
    min_class_count = df['sentiment_equal'].value_counts().min()
    return pd.concat([df[df['sentiment_equal'] == sentiment].sample(min_class_count, random_state=42) 
                      for sentiment in df['sentiment_equal'].unique()], ignore_index=True)


In [6]:
def predict(text, model, tokenizer, max_length):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=max_length).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    return probabilities.squeeze().cpu().numpy()

In [7]:
def ensemble_predict(text, models, tokenizers, max_lengths, method='voting'):
    preprocessed_text = preprocess_text(text)
    predictions = [predict(preprocessed_text, model, tokenizer, max_length) 
                   for model, tokenizer, max_length in zip(models, tokenizers, max_lengths)]
    predictions = np.array(predictions)
    
    if method == 'voting':
        votes = np.argmax(predictions, axis=1)
        return np.bincount(votes).argmax()
    elif method == 'averaging':
        avg_probs = np.mean(predictions, axis=0)
        return np.argmax(avg_probs)
    elif method == 'confidence':
        confidences = np.max(predictions, axis=1)
        weighted_votes = predictions * confidences[:, np.newaxis]
        return np.argmax(np.sum(weighted_votes, axis=0))

In [8]:
test_df = pd.read_csv("/kaggle/input/equal-neutral-submit-version/test_equal.csv")
test_df['comment'] = test_df['comment'].apply(preprocess_text)
undersampled_test_df = prepare_undersampled_data(test_df)

In [9]:
le = LabelEncoder()
undersampled_test_df['encoded_sentiment'] = le.fit_transform(undersampled_test_df['sentiment_equal'])

test_texts = undersampled_test_df['comment'].tolist()
test_labels_encoded = undersampled_test_df['encoded_sentiment'].tolist()

print(f"Undersampled test set size: {len(test_texts)}")

Undersampled test set size: 64134


In [10]:

def evaluate_ensemble(test_texts, test_labels, models, tokenizers, max_lengths, method='voting'):
    predictions = []
    for text in tqdm(test_texts, desc=f"Evaluating {method} ensemble"):
        predictions.append(ensemble_predict(text, models, tokenizers, max_lengths, method))
    
    accuracy = accuracy_score(test_labels, predictions)
    precision = precision_score(test_labels, predictions, average='weighted')
    recall = recall_score(test_labels, predictions, average='weighted')
    f1 = f1_score(test_labels, predictions, average='weighted')
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


In [11]:
methods = ['voting', 'averaging', 'confidence']

for method in methods:
    results = evaluate_ensemble(test_texts, test_labels_encoded, models, tokenizers, max_lengths, method=method)
    print(f"\n{method.capitalize()} Ensemble Results:")
    print(f"Accuracy: {results['accuracy']:.4f}")
    print(f"Precision: {results['precision']:.4f}")
    print(f"Recall: {results['recall']:.4f}")
    print(f"F1 Score: {results['f1']:.4f}")

Evaluating voting ensemble: 100%|██████████| 64134/64134 [46:38<00:00, 22.92it/s]



Voting Ensemble Results:
Accuracy: 0.7859
Precision: 0.7825
Recall: 0.7859
F1 Score: 0.7835


Evaluating averaging ensemble: 100%|██████████| 64134/64134 [46:47<00:00, 22.85it/s]



Averaging Ensemble Results:
Accuracy: 0.7868
Precision: 0.7836
Recall: 0.7868
F1 Score: 0.7845


Evaluating confidence ensemble: 100%|██████████| 64134/64134 [47:43<00:00, 22.40it/s]



Confidence Ensemble Results:
Accuracy: 0.7867
Precision: 0.7834
Recall: 0.7867
F1 Score: 0.7843


In [12]:
sample_text = test_texts[0] 
print("\nSample text:", sample_text)
print("Actual sentiment:", le.inverse_transform([test_labels_encoded[0]])[0])
for method in methods:
    prediction = ensemble_predict(sample_text, models, tokenizers, max_lengths, method=method)
    print(f"{method.capitalize()} prediction:", le.inverse_transform([prediction])[0])

def find_best_method(test_texts, test_labels, models, tokenizers, max_lengths):
    results = {method: evaluate_ensemble(test_texts, test_labels, models, tokenizers, max_lengths, method) for method in methods}
    best_method = max(results, key=lambda x: results[x]['f1'])
    return best_method, results[best_method]

best_method, best_results = find_best_method(test_texts, test_labels_encoded, models, tokenizers, max_lengths)
print(f"\nBest ensemble method: {best_method}")
print(f"Best results:")
print(f"Accuracy: {best_results['accuracy']:.4f}")
print(f"Precision: {best_results['precision']:.4f}")
print(f"Recall: {best_results['recall']:.4f}")
print(f"F1 Score: {best_results['f1']:.4f}")


Sample text: very handy! thank you very much!
Actual sentiment: Positive
Voting prediction: Positive
Averaging prediction: Positive
Confidence prediction: Positive


Evaluating confidence ensemble: 100%|██████████| 64134/64134 [44:40<00:00, 23.92it/s]



Best ensemble method: averaging
Best results:
Accuracy: 0.7868
Precision: 0.7836
Recall: 0.7868
F1 Score: 0.7845
